# Bi-Int Digital Twin — Evaluation Notebook

**Last updated:** 24 May 2026  
**Environment:** TwinCell (conda), Python 3.11, RDKit 2026.03, matplotlib 3.10  
**Hardware:** NVIDIA RTX 4000 Ada, 20,475 MiB VRAM, CUDA 13.0, TensorFlow 2.15.0

## Contents
1. Setup & imports
2. CCLE dataset summary
3. ChEMBL GNN pre-training curves (real — 10 epochs)
4. BiInt QSAR training curves (real — 4 epochs, random split)
5. BiInt Leave-Drug-Out results (real — 4 epochs)
6. Baseline comparison — all 5 models × 3 splits
7. GraphGA candidates — structures & drug-likeness
8. BRICS-DQN reward progression (5,000 episodes)
9. Summary dashboard

---

### Fixes applied (21–24 May 2026)

| Fix | Description | Status |
|-----|-------------|--------|
| P1 | Real drug SMILES: 3-level PubChem lookup → 201/266 matched | ✅ Done |
| P2 | Mutation alignment: full barcode match, sorted cells, assertions | ✅ Done |
| P3 | IC50 validation + 3 split modes (random / LDO / LCO) | ✅ Done |
| P4 | DQN reward: SA score + Lipinski hard penalty + Tanimoto diversity | ✅ Done |
| P5 | Baselines: Ridge/RF/MLP/XGBoost, ECFP4+omics, 3 splits | ✅ Done |
| P6 | TensorBoard + CSVLogger + EarlyStopping + gradient norm | ✅ Done |
| P7 | OOM: float32 + del DataFrames + NPZ cache + batch_size→8 | ✅ Done |
| P8 | O(n²) pandas loop on deleted DataFrame (3-day blocker) → O(n) numpy | ✅ Done |
| P9 | Epoch-2 crash: `drop_remainder=True` fixes dynamic batch dim | ✅ Done |
| P10 | IndexError after 20k subsampling in LDO/LCO splits | ✅ Done |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os, sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

try:
    from rdkit import Chem
    from rdkit.Chem import Draw, QED, Descriptors, AllChem, rdMolDescriptors, rdDepictor
    from rdkit.Chem.Draw import rdMolDraw2D
    from rdkit import RDLogger
    RDLogger.DisableLog('rdApp.*')
    from PIL import Image
    import io
    RDKIT_OK = True
except ImportError:
    print('RDKit not available — structure cells will be skipped')
    RDKIT_OK = False

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

# Paths (relative to notebooks/)
ROOT       = '..'
LOG_RANDOM = os.path.join(ROOT, 'logs/run_gpu_main/training_log.csv')
LOG_LDO    = os.path.join(ROOT, 'logs/run_ldo/training_log.csv')
GA_CSV     = os.path.join(ROOT, 'graphga_top_candidates.csv')
DQN_CSV    = os.path.join(ROOT, 'Dataset/brics_dqn_results.csv')
BL_CSV     = os.path.join(ROOT, 'Dataset/baseline_results.csv')
SMILES_CSV = os.path.join(ROOT, 'Dataset/ccle_drug_smiles.csv')

print('Setup OK — all paths configured')
for label, p in [('Random log', LOG_RANDOM), ('LDO log', LOG_LDO),
                 ('GA CSV', GA_CSV), ('DQN CSV', DQN_CSV),
                 ('Baselines', BL_CSV), ('SMILES map', SMILES_CSV)]:
    exists = '✓' if os.path.exists(p) else '✗ NOT FOUND'
    print(f'  {label:<15} {exists}  {p}')

## 2. CCLE Dataset Summary
Confirmed dimensions after P1–P3 fixes (21–24 May 2026).

In [ ]:
ccle_summary = {
    'Drugs (total IC50 matrix)':          266,
    'Cell lines (total IC50 matrix)':    1068,
    'Common cell lines (IC50+GEx+CNA)':   647,
    'Drugs with valid SMILES (PubChem)':  201,
    'Drugs missing SMILES':                65,
    'Valid triplets (drug, cell, IC50)': 103477,
    'GEx shape':                   '(647, 978)',
    'CNA shape':                   '(647, 426)',
    'Mutations shape':             '(647, 735)',
    'Mutation sparsity':               '0.844',
    'Mean mutations per cell line':    '115.0',
    'IC50 range (µM)':       '0.0001 — 400,374',
    'IC50 post-log1p mean':           '2.6747',
    'IC50 post-log1p std':            '1.8512',
    'Subsampled (RAM, seed=42)':       '20,000',
    'NPZ cache':    'omics_cache_gex978_cna426.npz',
}

df_ccle = pd.DataFrame(list(ccle_summary.items()), columns=['Metric', 'Value'])
print(df_ccle.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# SMILES coverage
axes[0].pie([201, 65], labels=['SMILES found\n(201)', 'Missing\n(65)'],
            colors=['#4CAF50', '#F44336'], autopct='%1.1f%%', startangle=90)
axes[0].set_title('Drug SMILES coverage\n(266 CCLE drugs, PubChem lookup)')

# Omics feature dimensions
axes[1].bar(['GEx\n(978)', 'CNA\n(426)', 'Mut\n(735)'], [978, 426, 735],
            color=['#2196F3', '#FF9800', '#9C27B0'], edgecolor='k', linewidth=0.6)
axes[1].set_ylabel('Feature dimensions')
axes[1].set_title('Omics features per cell line\n(647 common cell lines)')
for i, v in enumerate([978, 426, 735]):
    axes[1].text(i, v + 10, str(v), ha='center', fontsize=10)

# Triplet composition
labels = ['Train (random)\n~16,000', 'Val (random)\n~4,000']
axes[2].pie([16000, 4000], labels=labels, colors=['#42A5F5','#EF5350'],
            autopct='%1.0f%%', startangle=90)
axes[2].set_title('20k subsampled triplets\n(random split 80/20)')

plt.tight_layout()
plt.savefig('../figures/nb_01_ccle_summary.png', dpi=110, bbox_inches='tight')
plt.show()
print('\nFigure saved → figures/nb_01_ccle_summary.png')

## 3. ChEMBL GNN Pre-training (real — 10 epochs)
Self-supervised multi-task regression on 100k ChEMBL molecules.  
Targets: LogP, TPSA, MW, QED, HBD, HBA, NumRings, NumAromRings (all normalized).  
Best checkpoint: **epoch 9**, val RMSE = 0.2187, val loss = 0.0491.

In [ ]:
epochs     = list(range(1, 11))
train_rmse = [0.4886, 0.3462, 0.3034, 0.2796, 0.2649, 0.2511, 0.2410, 0.2322, 0.2247, 0.2192]
val_rmse   = [0.3598, 0.2972, 0.2671, 0.2471, 0.2467, 0.2327, 0.2323, 0.2300, 0.2187, 0.2215]
val_mae    = [0.2537, 0.2094, 0.1871, 0.1749, 0.1733, 0.1659, 0.1672, 0.1654, 0.1552, 0.1609]
val_loss   = [0.12948, 0.08831, 0.07136, 0.06104, 0.06084, 0.05415, 0.05397, 0.05291, 0.04784, 0.04890]

best_epoch = int(np.argmin(val_rmse)) + 1   # epoch 9

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('ChEMBL GNN Pre-training — Real Results (21 May 2026)', fontweight='bold')

axes[0].plot(epochs, train_rmse, 'o-', label='Train RMSE', color='#2196F3', linewidth=2)
axes[0].plot(epochs, val_rmse,   's-', label='Val RMSE',   color='#F44336', linewidth=2)
axes[0].plot(epochs, val_mae,    '^--', label='Val MAE',   color='#FF9800', alpha=0.8)
axes[0].annotate(f'Best: {min(val_rmse):.4f}\n(epoch {best_epoch})',
    xy=(best_epoch, min(val_rmse)),
    xytext=(best_epoch - 3.5, min(val_rmse) + 0.05),
    arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)
axes[0].axvline(8, color='gray', linestyle=':', linewidth=1, label='LR halved (1e-3→5e-4)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE / MAE (normalised targets)')
axes[0].set_title('Learning curves\n(RMSE=1.0 = predicting global mean)')
axes[0].legend(); axes[0].set_xticks(epochs); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, val_loss, 'o-', color='#9C27B0', linewidth=2, label='Val loss (MSE 8 targets)')
axes[1].axvline(best_epoch, color='green', linestyle='--', linewidth=1.5,
                label=f'Best checkpoint (ep {best_epoch})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Val loss (MSE)')
axes[1].set_title('Validation loss — ModelCheckpoint\n(saved at epoch 9, val_loss=0.0478)')
axes[1].legend(); axes[1].set_xticks(epochs); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/nb_02_chembl_pretrain.png', dpi=110, bbox_inches='tight')
plt.show()

df_pt = pd.DataFrame({'epoch': epochs, 'train_rmse': train_rmse,
                       'val_rmse': val_rmse, 'val_mae': val_mae, 'val_loss': val_loss})
df_pt['best'] = df_pt['epoch'] == best_epoch
print(df_pt.to_string(index=False))
print('\nInterpretation:')
print('  RMSE=0.219 on normalised targets = avg error of 0.22σ across 8 descriptors')
print('  RMSE=1.0 = predicting the mean (zero learning)')
print('  LR halved at epoch 8 (ReduceLROnPlateau, patience=2)')
print('  Transferred layers: node_embed, gcn_proj_1, ln1, node_proj, ln2')
print('  Weights → pretrained_weights/chembl_drug_encoder.weights.h5')
print('\nFigure saved → figures/nb_02_chembl_pretrain.png')

## 4. BiInt QSAR — Random Split (real — 4 epochs)

**Configuration:** `--loss-mode cross_entropy --split-mode random --batch-size 8`  
**Data:** 20k subsampled triplets, 201 drugs with real SMILES, 647 cell lines  
**Status:** 4 epochs complete. Epoch 5 terminated by GPU OOM (SelectV2, XLA, 17.6/20.5 GB VRAM)

| Epoch | Train RMSE | Val RMSE | Pearson r | Grad ‖∇‖ | KL/dim |
|-------|-----------|---------|-----------|---------|--------|
| 1 | 0.9595 | 0.8542 | 0.506 | 26.15 | 0.476 |
| 2 | 0.7674 | 0.8986 | 0.631 | 13.39 | 0.456 |
| 3 | 0.6693 | 0.5936 | 0.791 | 11.12 | 0.454 |
| **4** | **0.6058** | **0.5881** | **0.811** | 9.40 | 0.452 |

In [ ]:
df_qsar = pd.read_csv(LOG_RANDOM)
print(f'Loaded {len(df_qsar)} epochs from {LOG_RANDOM}')
print(df_qsar.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('BiInt QSAR — Random Split (real data, 24 May 2026)', fontweight='bold')

# RMSE
axes[0].plot(df_qsar['epoch'], df_qsar['train_rmse'], 'o-', label='Train RMSE',
             color='#2196F3', linewidth=2, markersize=7)
axes[0].plot(df_qsar['epoch'], df_qsar['val_rmse'],   's-', label='Val RMSE',
             color='#F44336', linewidth=2, markersize=7)
axes[0].fill_between(df_qsar['epoch'], df_qsar['train_rmse'], df_qsar['val_rmse'],
                     alpha=0.1, color='purple')
for ep, tr, va in zip(df_qsar['epoch'], df_qsar['train_rmse'], df_qsar['val_rmse']):
    axes[0].annotate(f'{tr:.3f}', (ep, tr), textcoords='offset points', xytext=(0,6),
                     fontsize=7, ha='center', color='#2196F3')
    axes[0].annotate(f'{va:.3f}', (ep, va), textcoords='offset points', xytext=(0,-12),
                     fontsize=7, ha='center', color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE (normalised log µM)')
axes[0].set_title('RMSE convergence\n(RMSE=1.0 → predicting global mean)')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Pearson r
axes[1].plot(df_qsar['epoch'], df_qsar['pearson_r'], 'o-',
             color='#4CAF50', linewidth=2, markersize=7)
axes[1].fill_between(df_qsar['epoch'], 0, df_qsar['pearson_r'], alpha=0.15, color='green')
for ep, r in zip(df_qsar['epoch'], df_qsar['pearson_r']):
    axes[1].annotate(f'{r:.3f}', (ep, r), textcoords='offset points', xytext=(0,6),
                     fontsize=8, ha='center')
axes[1].axhline(0.7, color='orange', linestyle='--', linewidth=1.2,
                label='Published CCLE floor (0.70)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Pearson r')
axes[1].set_title('Pearson r (validation)\nCompetitive with published models at r=0.70–0.85')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

# Gradient norm
axes[2].bar(df_qsar['epoch'], df_qsar['grad_norm'],
            color=['#FF7043','#FFA726','#FFCA28','#D4E157'], edgecolor='k', linewidth=0.6)
for ep, g in zip(df_qsar['epoch'], df_qsar['grad_norm']):
    axes[2].text(ep, g + 0.3, f'{g:.1f}', ha='center', fontsize=8)
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Gradient L2 norm')
axes[2].set_title('Gradient norm (L2)\n26.15→9.40: −64%, stable convergence')
axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/nb_03_qsar_random.png', dpi=110, bbox_inches='tight')
plt.show()

best = df_qsar.loc[df_qsar['val_rmse'].idxmin()]
print(f'\nBest epoch: {int(best["epoch"])}')
print(f'  Val RMSE   = {best["val_rmse"]:.4f}')
print(f'  Pearson r  = {best["pearson_r"]:.4f}')
print(f'  KL/dim     = {best["kl_loss"]:.4f}  (target: ~0.50, free_bits=0.5)')
print('\nInterpretation:')
print('  r=0.811 on random split = drug identity memorisation (not generalization)')
print('  KL=0.452 ≈ 0.5 free_bits → QuatVAE converged, no posterior collapse')
print('  Epoch 5 OOM: SelectV2 at 17.6/20.5 GB VRAM (XLA recompilation)')
print('\nFigure saved → figures/nb_03_qsar_random.png')

## 5. BiInt QSAR — Leave-Drug-Out Split (real — 4 epochs)

**Configuration:** `--loss-mode cross_entropy --split-mode leave_drug_out`  
**Split:** 171 train drugs | 30 val drugs → 16,832 train triplets | 3,168 val triplets  
**Key finding:** Overfitting from epoch 3 — best result at epoch 2 (r=0.316), **below XGBoost r=0.367**

| Epoch | Train RMSE | Val RMSE | Pearson r | Grad ‖∇‖ |
|-------|-----------|---------|-----------|---------|
| 1 | 0.9461 | 0.9978 | 0.253 | 32.11 |
| **2** | 0.7463 | **0.9834** | **0.316** | 13.21 |
| 3 | 0.6465 | 1.1579 | 0.209 | 10.60 |
| 4 | 0.6177 | 1.1441 | 0.257 | 9.79 |

In [ ]:
# LDO real data — hardcoded (matches logs/run_ldo/training_log.csv)
ldo_data = {
    'epoch':      [1,      2,      3,      4],
    'train_rmse': [0.9461, 0.7463, 0.6465, 0.6177],
    'val_rmse':   [0.9978, 0.9834, 1.1579, 1.1441],
    'pearson_r':  [0.253,  0.316,  0.209,  0.257],
    'grad_norm':  [32.11,  13.21,  10.60,  9.79],
}
df_ldo = pd.DataFrame(ldo_data)

# Load from CSV if available
if os.path.exists(LOG_LDO):
    df_ldo = pd.read_csv(LOG_LDO)
    print(f'Loaded from {LOG_LDO}')
print(df_ldo.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('BiInt QSAR — Leave-Drug-Out Split (real, 24 May 2026)', fontweight='bold')

# RMSE + overfitting zone
axes[0].plot(df_ldo['epoch'], df_ldo['train_rmse'], 'o-', label='Train RMSE',
             color='#2196F3', linewidth=2, markersize=7)
axes[0].plot(df_ldo['epoch'], df_ldo['val_rmse'],   's-', label='Val RMSE',
             color='#F44336', linewidth=2, markersize=7)
axes[0].axvspan(2.5, 4.5, alpha=0.08, color='red', label='Overfitting zone')
axes[0].axvline(2, color='green', linestyle='--', linewidth=1.5, label='Best (ep 2, r=0.316)')
for ep, tr, va in zip(df_ldo['epoch'], df_ldo['train_rmse'], df_ldo['val_rmse']):
    axes[0].annotate(f'{tr:.3f}', (ep, tr), textcoords='offset points', xytext=(0,6),
                     fontsize=7, ha='center', color='#2196F3')
    axes[0].annotate(f'{va:.3f}', (ep, va), textcoords='offset points', xytext=(0,-12),
                     fontsize=7, ha='center', color='#F44336')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('RMSE (normalised log µM)')
axes[0].set_title('LDO RMSE — overfitting from epoch 3\n(val 0.983→1.158, train continues falling)')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Pearson r comparison with baselines
models   = ['Ridge\nECFP4', 'Ridge\nOmics', 'RF', 'MLP', 'XGBoost', 'Bi-Int\nep.2']
ldo_r    = [0.286, 0.295, 0.174, 0.349, 0.367, 0.316]
colors_b = ['#90CAF9','#B0BEC5','#B0BEC5','#B0BEC5','#EF9A9A','#A5D6A7']
bars = axes[1].bar(models, ldo_r, color=colors_b, edgecolor='k', linewidth=0.7)
axes[1].axhline(0.367, color='red', linestyle='--', linewidth=1.5, label='XGBoost (best classical, 0.367)')
axes[1].bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
axes[1].set_ylabel('Pearson r (LDO validation)')
axes[1].set_title('Leave-Drug-Out: Bi-Int vs classical baselines\n(r=0.316 vs XGBoost r=0.367, −0.051)')
axes[1].legend(fontsize=8); axes[1].set_ylim(0, 0.5); axes[1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/nb_04_qsar_ldo.png', dpi=110, bbox_inches='tight')
plt.show()

print('\nKey finding:')
print('  Bi-Int LDO r=0.316 < XGBoost r=0.367 (−0.051)')
print('  Overfitting from epoch 3: val RMSE +17.7%, Pearson r −0.107')
print('  Root cause candidates:')
print('    1. 20k subsampling (only 16,832 LDO train triplets)')
print('    2. No early stopping (val RMSE was still best at epoch 2)')
print('    3. 9.2M params >> 16k samples → regularisation needed')
print('\nFigure saved → figures/nb_04_qsar_ldo.png')

In [ ]:
# Cell 5 — Drug SMILES mapping summary
smiles_path = '../Dataset/ccle_drug_smiles.csv'

if not os.path.exists(smiles_path):
    print(f'ccle_drug_smiles.csv not found. Run: python fetch_drug_smiles.py')
else:
    df_smi = pd.read_csv(smiles_path)
    n_total  = len(df_smi)
    n_mapped = df_smi['smiles'].notna().sum()
    n_missed = n_total - n_mapped

    print(f'Drug SMILES mapping  ({smiles_path})')
    print(f'  Total  : {n_total}')
    print(f'  Mapped : {n_mapped} ({100*n_mapped/n_total:.1f}%)')
    print(f'  Missing: {n_missed} ({100*n_missed/n_total:.1f}%)')

    if 'source' in df_smi.columns:
        print('\nSource breakdown:')
        print(df_smi['source'].value_counts(dropna=False).to_string())

    mapped = df_smi[df_smi['smiles'].notna()]
    print('\n5 example mappings:')
    for _, row in mapped.head(5).iterrows():
        smi = str(row['smiles'])[:60] + ('...' if len(str(row['smiles'])) > 60 else '')
        print(f"  {row['drug_name']:<35s} -> {smi}")

    missed_list = df_smi[df_smi['smiles'].isna()]['drug_name'].tolist()
    if missed_list:
        print(f'\nUnmapped drugs ({len(missed_list)}):')
        for m in missed_list[:20]:
            print(f'  - {m}')
        if len(missed_list) > 20:
            print(f'  ... and {len(missed_list)-20} more')

## 8. BRICS-DQN Reward Progression (5,000 real episodes)

**Algorithm:** Double DQN, BRICS fragment-assembly MDP, vocabulary from ChEMBL 10k  
**Results:** Best reward = 6.124, validity = 60.5%, 5,000 episodes  
**Source:** `Dataset/brics_dqn_results.csv`

In [ ]:
df_dqn = pd.read_csv(DQN_CSV)
valid_mask = df_dqn['valid'].astype(str).str.lower() == 'true'
rewards    = df_dqn['reward'].values
episodes   = df_dqn['episode'].values

rolling = pd.Series(rewards).rolling(50, min_periods=1).mean().values
window  = 100
n_bins  = len(rewards) // window
bin_means = [rewards[i*window:(i+1)*window].mean() for i in range(n_bins)]
bin_stds  = [rewards[i*window:(i+1)*window].std()  for i in range(n_bins)]
bin_valid = [valid_mask.values[i*window:(i+1)*window].mean()*100 for i in range(n_bins)]
bin_x     = [(i + 0.5) * window for i in range(n_bins)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('BRICS-DQN — Reward over 5,000 Episodes (real data, 24 May 2026)',
             fontweight='bold')

# Raw rewards
axes[0].scatter(episodes[valid_mask],  rewards[valid_mask],
                s=1.5, alpha=0.2, color='#42A5F5', label='Valid')
axes[0].scatter(episodes[~valid_mask], rewards[~valid_mask],
                s=1.5, alpha=0.1, color='#EF5350', label='Invalid')
axes[0].plot(episodes, rolling, 'k-', linewidth=1.8, label='Rolling mean (w=50)', zorder=5)
best_idx = rewards.argmax()
axes[0].annotate(f'Best: {rewards.max():.3f}',
    xy=(episodes[best_idx], rewards[best_idx]),
    xytext=(episodes[best_idx]+300, rewards[best_idx]-0.5),
    arrowprops=dict(arrowstyle='->', color='red'), color='red', fontsize=8)
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].set_title(f'Raw rewards + rolling mean\n(best={rewards.max():.3f}, mean={rewards.mean():.3f})')
axes[0].legend(markerscale=4, fontsize=8); axes[0].grid(alpha=0.2)

# 100-episode blocks
axes[1].plot(bin_x, bin_means, 'o-', color='#1565C0', linewidth=2, markersize=4)
axes[1].fill_between(bin_x,
    [m-s for m,s in zip(bin_means, bin_stds)],
    [m+s for m,s in zip(bin_means, bin_stds)],
    alpha=0.2, color='blue', label='±1 std')
axes[1].axhline(np.mean(bin_means), color='red', linestyle='--', linewidth=1.2,
                label=f'Global mean = {np.mean(bin_means):.2f}')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Mean reward / 100 ep')
axes[1].set_title('Reward per 100-episode block\n(no clear upward trend → not converged)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Validity rate
axes[2].plot(bin_x, bin_valid, '-^', color='#2E7D32', linewidth=2, markersize=4)
axes[2].fill_between(bin_x, bin_valid, alpha=0.15, color='green')
axes[2].axhline(valid_mask.mean()*100, color='red', linestyle='--', linewidth=1.5,
                label=f'Global: {valid_mask.mean()*100:.1f}%')
axes[2].set_xlabel('Episode'); axes[2].set_ylabel('% Valid SMILES')
axes[2].set_title('SMILES validity per 100-episode block\n(60.5% global — stable)')
axes[2].set_ylim(0, 105); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/nb_07_dqn_reward.png', dpi=110, bbox_inches='tight')
plt.show()

print(f'DQN statistics:')
print(f'  Total episodes : {len(df_dqn):,}')
print(f'  Valid SMILES   : {valid_mask.sum():,} / {len(df_dqn):,} ({valid_mask.mean()*100:.1f}%)')
print(f'  Best reward    : {rewards.max():.4f}  (ep {episodes[rewards.argmax()]})')
print(f'  Mean reward    : {rewards.mean():.4f}')
print(f'  Mean (valid)   : {rewards[valid_mask].mean():.4f}')
print()
print('Top 5 episodes by reward:')
top5_idx = rewards.argsort()[-5:][::-1]
for i in top5_idx:
    print(f'  ep={int(episodes[i]):<5} reward={rewards[i]:.3f}  {df_dqn["smiles"].iloc[i][:70]}')
print('\nFigure saved → figures/nb_07_dqn_reward.png')

## 6. Baseline Comparison — All 5 Models × 3 Splits

**Features:** ECFP4 Morgan r=2 (2048 bits) + GEx (978) + CNA (426) = 4,452 dims  
**Models:** Ridge, RF (50 trees), MLP (256→128), XGBoost (100 trees)  
**Splits:** Random (80/20), Leave-Drug-Out (30 val drugs), Leave-Cell-Out (129 val cell lines)

In [ ]:
# Full baseline results — real data (24 May 2026)
baseline_data = {
    'Model': ['Ridge (ECFP4+omics)', 'Ridge (omics only)', 'RF (50 trees)',
              'MLP (256→128)', 'XGBoost (100 trees)',
              'Ridge (ECFP4+omics)', 'Ridge (omics only)', 'RF (50 trees)',
              'MLP (256→128)', 'XGBoost (100 trees)',
              'Ridge (ECFP4+omics)', 'Ridge (omics only)', 'RF (50 trees)',
              'MLP (256→128)', 'XGBoost (100 trees)'],
    'Split':  (['Random'] * 5) + (['Leave-Drug-Out'] * 5) + (['Leave-Cell-Out'] * 5),
    'RMSE':   [0.508, 0.971, 0.824, 0.477, 0.548,
               1.033, 0.956, 1.015, 0.975, 0.938,
               0.601, 1.020, 0.826, 0.817, 0.580],
    'R2':     [0.746, 0.070, 0.331, 0.776, 0.704,
               -0.065, 0.087, -0.029, 0.050, 0.121,
               0.642, -0.029, 0.326, 0.340, 0.668],
    'Pearson_r': [0.864, 0.265, 0.584, 0.881, 0.849,
                  0.286, 0.295, 0.174, 0.349, 0.367,
                  0.803, 0.095, 0.579, 0.676, 0.824],
}
df_bl = pd.DataFrame(baseline_data)

# Add BiInt rows
biint_rows = pd.DataFrame({
    'Model':     ['Bi-Int (epoch 4)', 'Bi-Int (epoch 2)'],
    'Split':     ['Random', 'Leave-Drug-Out'],
    'RMSE':      [0.5881, 0.9834],
    'R2':        [float('nan'), float('nan')],
    'Pearson_r': [0.811, 0.316],
})
df_all = pd.concat([df_bl, biint_rows], ignore_index=True)

print('=== Pearson r — All Models × All Splits ===')
pivot = df_all.pivot_table(index='Model', columns='Split', values='Pearson_r', aggfunc='first')
# reorder columns
col_order = [c for c in ['Random', 'Leave-Drug-Out', 'Leave-Cell-Out'] if c in pivot.columns]
print(pivot[col_order].to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Baselines vs Bi-Int — Pearson r by Split (real data, 24 May 2026)',
             fontweight='bold', fontsize=12)

splits = ['Random', 'Leave-Drug-Out', 'Leave-Cell-Out']
titles = ['Random Split (memorisation)\nAll drugs seen in train',
          'Leave-Drug-Out\n30 unseen drugs in val',
          'Leave-Cell-Out\n129 unseen cell lines in val']
palettes = [['#42A5F5','#B0BEC5','#B0BEC5','#B0BEC5','#B0BEC5','#FF7043'],
            ['#42A5F5','#B0BEC5','#B0BEC5','#B0BEC5','#EF9A9A','#A5D6A7'],
            ['#42A5F5','#B0BEC5','#B0BEC5','#B0BEC5','#EF9A9A']]

for ax, split, title, pal in zip(axes, splits, titles, palettes):
    sub = df_all[df_all['Split'] == split].sort_values('Pearson_r', ascending=False)
    bars = ax.bar(range(len(sub)), sub['Pearson_r'], color=pal[:len(sub)],
                  edgecolor='k', linewidth=0.6)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=8)
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels([m.replace(' (ECFP4+omics)','').replace(' (omics only)','\n(omics)')
                        for m in sub['Model']], rotation=20, ha='right', fontsize=8)
    ax.set_ylabel('Pearson r'); ax.set_title(title)
    ax.set_ylim(0, 1.05); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/nb_05_baselines.png', dpi=110, bbox_inches='tight')
plt.show()

print('\nKey findings:')
print('  Random split: MLP r=0.881 best (drug identity memorisation artefact)')
print('  Leave-Drug-Out: XGBoost r=0.367 best classical; Bi-Int r=0.316 (−0.051)')
print('  Leave-Cell-Out: XGBoost r=0.824 — most stable model across all splits')
print('  Ridge ECFP4+omics: R²=−0.065 in LDO (worse than predicting the mean)')
print('  Ridge omics only > Ridge ECFP4+omics in LDO: ECFP4 adds noise for unseen drugs')
print('\nFigure saved → figures/nb_05_baselines.png')

## 7. GraphGA Candidates — Structures & Drug-Likeness

10 drug candidates generated by the GraphGA evolutionary optimizer  
guided by the Bi-Int IC50 predictor. Source: `graphga_top_candidates.csv`

In [ ]:
df_ga = pd.read_csv(GA_CSV)
print(f'Loaded {len(df_ga)} candidates from {GA_CSV}')
print(df_ga[['rank','qed','sa','mw','logp','composite','pains']].to_string(index=False))

if RDKIT_OK:
    # 2D molecular grid
    mols, legends = [], []
    for _, row in df_ga.iterrows():
        mol = Chem.MolFromSmiles(row['smiles'])
        if mol:
            rdDepictor.Compute2DCoords(mol)
            mols.append(mol)
            legends.append(f"#{int(row['rank'])}  QED={row['qed']:.3f}\n"
                           f"MW={row['mw']:.0f}  LogP={row['logp']:.2f}")

    img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(360, 290),
                               legends=legends, returnPNG=False)
    img.save('../figures/nb_06_ga_structures.png')
    display(img)
    print('Figure saved → figures/nb_06_ga_structures.png')

    # QED / MW / LogP bar charts
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('GraphGA Candidates — Drug-Likeness Profile (real data)', fontweight='bold')
    labels = [f"#{int(r)}" for r in df_ga['rank']]

    ax = axes[0]
    colors = ['gold' if q >= 0.9 else '#42A5F5' if q >= 0.8 else '#EF9A9A' for q in df_ga['qed']]
    bars = ax.bar(labels, df_ga['qed'], color=colors, edgecolor='k', linewidth=0.6)
    ax.axhline(0.7, color='red',  linestyle='--', linewidth=1.5, label='Drug-like ≥0.7')
    ax.axhline(0.9, color='gold', linestyle=':',  linewidth=1.5, label='Excellent ≥0.9')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
    ax.set_ylabel('QED'); ax.set_title('Drug-Likeness (QED)\nMean = {:.3f}'.format(df_ga['qed'].mean()))
    ax.legend(fontsize=8); ax.set_ylim(0, 1.1); ax.grid(alpha=0.3, axis='y')

    ax = axes[1]
    ax.bar(labels, df_ga['mw'], color='#66BB6A', edgecolor='k', linewidth=0.6)
    ax.axhline(500, color='red', linestyle='--', linewidth=1.5, label='Lipinski MW≤500')
    ax.set_ylabel('MW (Da)'); ax.set_title(f'Molecular Weight\nRange {df_ga["mw"].min():.0f}–{df_ga["mw"].max():.0f} Da')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

    ax = axes[2]
    colors_l = ['#42A5F5' if l <= 5 else '#EF5350' for l in df_ga['logp']]
    ax.bar(labels, df_ga['logp'], color=colors_l, edgecolor='k', linewidth=0.6)
    ax.axhline(5, color='red', linestyle='--', linewidth=1.5, label='Lipinski LogP≤5')
    ax.set_ylabel('LogP'); ax.set_title(f'LogP (lipophilicity)\nMean {df_ga["logp"].mean():.2f} (optimum 1–3)')
    ax.legend(fontsize=8); ax.grid(alpha=0.3, axis='y')

    plt.tight_layout()
    plt.savefig('../figures/nb_06_ga_properties.png', dpi=110, bbox_inches='tight')
    plt.show()
    print('Figure saved → figures/nb_06_ga_properties.png')
    print(f'\nAll {len(df_ga)} candidates pass Lipinski Ro5: ' +
          ('YES' if all(df_ga['mw']<=500) and all(df_ga['logp']<=5) else 'NO'))
    print(f'PAINS alerts: {df_ga["pains"].sum()} / {len(df_ga)}')
else:
    print('RDKit not available — skipping structure drawing')

## 9. Summary Dashboard — All Results in One View

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('#f8f9fa')
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.5, wspace=0.4)
fig.suptitle('Bi-Int Digital Twin — Complete Results Dashboard (24 May 2026)',
             fontsize=14, fontweight='bold', y=0.98)

# ── QSAR Random RMSE ────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 0:2])
df_q = pd.read_csv(LOG_RANDOM)
ax.plot(df_q['epoch'], df_q['train_rmse'], 'o-', label='Train', color='#2196F3', lw=2)
ax.plot(df_q['epoch'], df_q['val_rmse'],   's-', label='Val',   color='#F44336', lw=2)
ax.set_title('QSAR RMSE (random split)'); ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
for ep, tr, va in zip(df_q['epoch'], df_q['train_rmse'], df_q['val_rmse']):
    ax.text(ep, tr+0.01, f'{tr:.3f}', ha='center', fontsize=7, color='#2196F3')
    ax.text(ep, va-0.04, f'{va:.3f}', ha='center', fontsize=7, color='#F44336')

# ── QSAR Pearson r ─────────────────────────────────────────────
ax = fig.add_subplot(gs[0, 2])
ax.plot(df_q['epoch'], df_q['pearson_r'], 'o-', color='#4CAF50', lw=2, markersize=7)
for ep, r in zip(df_q['epoch'], df_q['pearson_r']):
    ax.text(ep, r+0.02, f'{r:.3f}', ha='center', fontsize=8)
ax.set_title('Pearson r (random val)'); ax.set_xlabel('Epoch'); ax.set_ylabel('r')
ax.set_ylim(0, 1); ax.grid(alpha=0.3)

# ── Baseline comparison ─────────────────────────────────────────
ax = fig.add_subplot(gs[0, 3])
models_ldo = ['Ridge\nECFP4', 'Ridge\nOmics', 'RF', 'MLP', 'XGB', 'Bi-Int']
r_ldo      = [0.286, 0.295, 0.174, 0.349, 0.367, 0.316]
colors_c   = ['#90CAF9','#B0BEC5','#B0BEC5','#B0BEC5','#EF9A9A','#A5D6A7']
bars = ax.bar(models_ldo, r_ldo, color=colors_c, edgecolor='k', linewidth=0.6)
ax.axhline(0.367, color='red', linestyle='--', linewidth=1.2)
ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=7)
ax.set_title('LDO Pearson r'); ax.set_ylabel('r'); ax.set_ylim(0, 0.5)
ax.tick_params(labelsize=7); ax.grid(alpha=0.3, axis='y')

# ── DQN reward ─────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0:2])
ax.scatter(episodes[valid_mask], rewards[valid_mask], s=1, alpha=0.15, color='#42A5F5')
ax.plot(episodes, rolling, 'k-', linewidth=1.5, label='Rolling mean (w=50)')
ax.set_title(f'DQN Reward (5,000 ep) — best={rewards.max():.3f}, valid={valid_mask.mean()*100:.1f}%')
ax.set_xlabel('Episode'); ax.set_ylabel('Reward')
ax.legend(fontsize=8); ax.grid(alpha=0.2)

# ── QED bar chart ───────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 2])
labels_ga = [f"#{int(r)}" for r in df_ga['rank']]
colors_q  = ['gold' if q >= 0.9 else '#42A5F5' if q >= 0.8 else '#EF9A9A'
             for q in df_ga['qed']]
ax.bar(labels_ga, df_ga['qed'], color=colors_q, edgecolor='k', linewidth=0.5)
ax.axhline(0.7, color='red', linestyle='--', linewidth=1.2, label='≥0.7')
ax.set_title('QED — GraphGA candidates'); ax.set_ylabel('QED')
ax.set_ylim(0, 1.1); ax.legend(fontsize=7); ax.grid(alpha=0.3, axis='y')
ax.tick_params(labelsize=8)

# ── Metrics table ────────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 3])
ax.axis('off')
table_data = [
    ['Metric', 'Value'],
    ['QSAR epochs done',   '4 (OOM ep.5)'],
    ['Val RMSE (random)',  '0.5881'],
    ['Pearson r (random)', '0.811'],
    ['Pearson r (LDO)',    '0.316'],
    ['XGBoost LDO r',      '0.367 (best baseline)'],
    ['DQN episodes',       '5,000'],
    ['DQN best reward',    '6.124 / 6.5 (94%)'],
    ['SMILES validity',    '60.5%'],
    ['GA candidates',      '10 (all Lipinski OK)'],
    ['QED mean',           f'{df_ga["qed"].mean():.3f}'],
    ['PAINS alerts',       '0 / 10'],
]
tbl = ax.table(cellText=table_data[1:], colLabels=table_data[0],
               cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(8)
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#1565C0'); cell.set_text_props(color='white', fontweight='bold')
    elif row % 2 == 0:
        cell.set_facecolor('#E3F2FD')
    cell.set_edgecolor('#B0BEC5')
ax.set_title('Summary Metrics', fontweight='bold', fontsize=10)

# ── ChEMBL pre-train ─────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0:2])
epochs_pt  = list(range(1, 11))
val_rmse_pt = [0.3598, 0.2972, 0.2671, 0.2471, 0.2467, 0.2327, 0.2323, 0.2300, 0.2187, 0.2215]
ax.plot(epochs_pt, val_rmse_pt, 'o-', color='#9C27B0', lw=2, markersize=6)
ax.axvline(9, color='green', linestyle='--', lw=1.5, label='Best (ep9, 0.2187)')
ax.set_title('ChEMBL GNN pre-training (val RMSE, 10 epochs)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Val RMSE (normalised)')
ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_xticks(epochs_pt)

# ── MW distribution ──────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 2])
ax.bar(labels_ga, df_ga['mw'], color='#66BB6A', edgecolor='k', linewidth=0.5)
ax.axhline(500, color='red', linestyle='--', lw=1.2, label='Ro5 ≤500 Da')
ax.set_title('MW — GraphGA candidates'); ax.set_ylabel('MW (Da)')
ax.legend(fontsize=7); ax.tick_params(labelsize=8); ax.grid(alpha=0.3, axis='y')

# ── Validity per block ───────────────────────────────────────────
ax = fig.add_subplot(gs[2, 3])
ax.plot(bin_x, bin_valid, '-^', color='#2E7D32', lw=2, markersize=3)
ax.fill_between(bin_x, bin_valid, alpha=0.15, color='green')
ax.axhline(60.5, color='red', linestyle='--', lw=1.2, label='Mean 60.5%')
ax.set_title('DQN validity / 100-ep block'); ax.set_ylabel('%')
ax.set_ylim(0, 105); ax.legend(fontsize=7); ax.grid(alpha=0.3)

plt.savefig('../figures/nb_08_dashboard.png', dpi=110, bbox_inches='tight', facecolor='#f8f9fa')
plt.show()
print('Figure saved → figures/nb_08_dashboard.png')